# 3장 1강: 교차표와 카이제곱 독립성 검정 이론 — 실습문제

## 실습 목표

- 두 범주형 변수의 교차표를 구성하고 관측빈도를 해석할 수 있다.
- 행 합계와 열 합계를 이용해 기대빈도를 직접 계산할 수 있다.
- 관측빈도와 기대빈도로 카이제곱 통계량과 자유도를 계산할 수 있다.
- 카이제곱 독립성 검정을 수행하고 기대빈도 조건을 확인할 수 있다.
- p-value와 범주별 비율을 함께 사용하여 변수 간 관련성을 해석할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas, NumPy
- scipy.stats
- `ames_housing.csv`

| 컬럼 | 의미 |
|---|---|
| `OverallQual` | 주택의 전반적인 품질 점수 |
| `YearBuilt` | 건축연도 |
| `CentralAir` | 중앙 냉방시설 유무 |
| `KitchenQual` | 주방 품질 |
| `PavedDrive` | 진입로 포장 상태 |

> 모든 검정의 유의수준은 `α = 0.05`입니다.  
> `chi2_contingency()`에서는 강의자료와 동일하게 `correction=False`를 사용합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.
import pandas as pd

df = pd.read_csv(r'C:\Machine_Learning_worksheet\Data\ames_housing.csv', encoding = 'utf-8-sig')


# 1. 상위 5개 행 출력
print(df.head(5))

# 2. 상위 5개 컬럼 이름 출력 (슬라이싱 사용)
print(df.columns[:5])

# 3. 컬럼별 결측치 개수 확인
print(df.isna().sum())

   SalePrice  GrLivArea  LotArea  OverallQual KitchenQual CentralAir  \
0     208500       1710     8450            7          Gd          Y   
1     181500       1262     9600            6          TA          Y   
2     223500       1786    11250            7          Gd          Y   
3     140000       1717     9550            7          Gd          Y   
4     250000       2198    14260            8          Gd          Y   

  HeatingQC PavedDrive Neighborhood  YearBuilt  
0        Ex          Y      CollgCr       2003  
1        Ex          Y      Veenker       1976  
2        Ex          Y      CollgCr       2001  
3        Gd          Y      Crawfor       1915  
4        Ex          Y      NoRidge       2000  
Index(['SalePrice', 'GrLivArea', 'LotArea', 'OverallQual', 'KitchenQual'], dtype='str')
SalePrice       0
GrLivArea       0
LotArea         0
OverallQual     0
KitchenQual     0
CentralAir      0
HeatingQC       0
PavedDrive      0
Neighborhood    0
YearBuilt       0
dtype

---

## 필수 1. 교차표와 카이제곱 통계량 직접 계산

### 문제 1-1. 주택 품질 구간과 중앙 냉방시설의 관계

#### 문제 설명

`OverallQual`을 세 구간으로 나눈 뒤 중앙 냉방시설 유무와의 관계를 확인합니다.

- 낮음: 1~5점
- 보통: 6~7점
- 높음: 8~10점

#### 요구사항

1. `pd.cut()`을 이용해 위 기준으로 `QualityGroup`을 만드세요.
2. 행에는 `QualityGroup`, 열에는 `CentralAir`가 오도록 합계 없는 교차표를 만드세요.
3. `margins=True`인 교차표도 별도로 만들어 행·열 합계를 확인하세요.
4. 합계가 없는 교차표를 NumPy 배열 `observed`로 변환하세요.
5. 행 합계, 열 합계, 전체 합계를 계산하세요.
6. `(행 합계 × 열 합계) / 전체 합계`로 기대빈도를 직접 계산하세요.
7. `Σ(관측-기대)²/기대`로 카이제곱 통계량을 직접 계산하세요.
8. `(행 수-1) × (열 수-1)`로 자유도를 계산하세요.
9. 모든 기대빈도가 5 이상인지 확인하세요.
10. `stats.chi2_contingency(..., correction=False)` 결과와 직접 계산한 값을 비교하세요.
11. p-value를 이용해 두 변수가 관련 있는지 판단하세요.

#### 해석 질문

**Q1.** 교차표 각 칸의 숫자는 무엇을 의미하나요?  
**Q2.** 기대빈도는 어떤 가정 아래 계산되는 값인가요?  
**Q3.** 직접 계산한 카이제곱 통계량과 함수의 결과는 일치하나요?  
**Q4.** 주택 품질 구간과 중앙 냉방시설 유무는 서로 독립이라고 볼 수 있나요?

#### 제출 결과

- 관측빈도 교차표와 주변합
- 기대빈도
- 수동 계산한 카이제곱 통계량과 자유도
- 기대빈도 조건 확인
- 함수 결과와의 비교
- 독립성 판단
- Q1~Q4 답변

In [6]:
"""
QualityGroup x CentralAir 카이제곱 독립성 검정
"""

import numpy as np
import pandas as pd
from scipy import stats


# ---------------------------------------------------------------
# 1. pd.cut() 으로 QualityGroup 만들기
#    (0,5] = Low(1~5), (5,7] = Mid(6~7), (7,10] = High(8~10)
# ---------------------------------------------------------------
bins = [0, 5, 7, 10]
labels = ["Low", "Mid", "High"]

df["QualityGroup"] = pd.cut(df["OverallQual"], bins=bins, labels=labels)

print("[1] QualityGroup 분포")
print(df["QualityGroup"].value_counts().sort_index(), "\n")

# ---------------------------------------------------------------
# 2. 합계 없는 교차표 (행: QualityGroup, 열: CentralAir)
# ---------------------------------------------------------------
table = pd.crosstab(df["QualityGroup"], df["CentralAir"])

print("[2] 교차표 (합계 없음)")
print(table, "\n")

# ---------------------------------------------------------------
# 3. margins=True 교차표로 행·열 합계 확인
# ---------------------------------------------------------------
table_margins = pd.crosstab(df["QualityGroup"], df["CentralAir"], margins=True)

print("[3] 교차표 (margins=True)")
print(table_margins, "\n")

# ---------------------------------------------------------------
# 4. NumPy 배열 observed 로 변환
# ---------------------------------------------------------------
observed = table.to_numpy()

print("[4] observed 배열")
print(observed, "\n")

# ---------------------------------------------------------------
# 5. 행 합계, 열 합계, 전체 합계
# ---------------------------------------------------------------
row_sums = observed.sum(axis=1)   # 행 합계
col_sums = observed.sum(axis=0)   # 열 합계
total = observed.sum()            # 전체 합계

print("[5] 행 합계:", row_sums)
print("    열 합계:", col_sums)
print("    전체 합계:", total, "\n")

# ---------------------------------------------------------------
# 6. 기대빈도 = (행 합계 x 열 합계) / 전체 합계
# ---------------------------------------------------------------
expected = np.outer(row_sums, col_sums) / total

print("[6] 기대빈도")
print(np.round(expected, 3), "\n")

# ---------------------------------------------------------------
# 7. 카이제곱 통계량 = Σ (관측 - 기대)² / 기대
# ---------------------------------------------------------------
chi2_manual = ((observed - expected) ** 2 / expected).sum()

print(f"[7] 카이제곱 통계량 (직접 계산) = {chi2_manual:.6f}\n")

# ---------------------------------------------------------------
# 8. 자유도 = (행 수 - 1) x (열 수 - 1)
# ---------------------------------------------------------------
n_rows, n_cols = observed.shape
dof_manual = (n_rows - 1) * (n_cols - 1)

print(f"[8] 자유도 = ({n_rows}-1) x ({n_cols}-1) = {dof_manual}\n")

# ---------------------------------------------------------------
# 9. 모든 기대빈도가 5 이상인지 확인
# ---------------------------------------------------------------
all_ge_5 = bool((expected >= 5).all())
min_expected = expected.min()

print(f"[9] 최소 기대빈도 = {min_expected:.3f}")
print(f"    모든 기대빈도 >= 5 ? {all_ge_5}")
if not all_ge_5:
    print("    → 기대빈도 5 미만 셀이 있어 카이제곱 근사가 부정확할 수 있습니다.")
print()

# ---------------------------------------------------------------
# 10. scipy 결과와 비교 (correction=False: 연속성 보정 끄기)
# ---------------------------------------------------------------
chi2_scipy, p_scipy, dof_scipy, expected_scipy = stats.chi2_contingency(
    observed, correction=False
)

print("[10] scipy 결과와 비교")
print(f"     카이제곱  직접={chi2_manual:.6f} | scipy={chi2_scipy:.6f} | "
      f"일치={np.isclose(chi2_manual, chi2_scipy)}")
print(f"     자유도    직접={dof_manual} | scipy={dof_scipy} | "
      f"일치={dof_manual == dof_scipy}")
print(f"     기대빈도  일치={np.allclose(expected, expected_scipy)}\n")

# ---------------------------------------------------------------
# 11. p-value 로 두 변수의 관련성 판단
# ---------------------------------------------------------------
p_manual = stats.chi2.sf(chi2_manual, dof_manual)   # sf = 1 - cdf
alpha = 0.05

print("[11] 검정 결과")
print(f"     p-value (직접) = {p_manual:.6g}")
print(f"     p-value (scipy) = {p_scipy:.6g}")
print(f"     유의수준 alpha = {alpha}")

if p_manual < alpha:
    print(f"     → p < {alpha} 이므로 귀무가설(두 변수는 독립이다)을 기각합니다.")
    print("       QualityGroup 과 CentralAir 는 서로 관련이 있습니다.")
else:
    print(f"     → p >= {alpha} 이므로 귀무가설을 기각하지 못합니다.")
    print("       두 변수가 관련 있다고 보기 어렵습니다.")

[1] QualityGroup 분포
QualityGroup
Low     538
Mid     693
High    229
Name: count, dtype: int64 

[2] 교차표 (합계 없음)
CentralAir     N    Y
QualityGroup         
Low           71  467
Mid           23  670
High           1  228 

[3] 교차표 (margins=True)
CentralAir     N     Y   All
QualityGroup                
Low           71   467   538
Mid           23   670   693
High           1   228   229
All           95  1365  1460 

[4] observed 배열
[[ 71 467]
 [ 23 670]
 [  1 228]] 

[5] 행 합계: [538 693 229]
    열 합계: [  95 1365]
    전체 합계: 1460 

[6] 기대빈도
[[ 35.007 502.993]
 [ 45.092 647.908]
 [ 14.901 214.099]] 

[7] 카이제곱 통계량 (직접 계산) = 65.030381

[8] 자유도 = (3-1) x (2-1) = 2

[9] 최소 기대빈도 = 14.901
    모든 기대빈도 >= 5 ? True

[10] scipy 결과와 비교
     카이제곱  직접=65.030381 | scipy=65.030381 | 일치=True
     자유도    직접=2 | scipy=2 | 일치=True
     기대빈도  일치=True

[11] 검정 결과
     p-value (직접) = 7.5654e-15
     p-value (scipy) = 7.5654e-15
     유의수준 alpha = 0.05
     → p < 0.05 이므로 귀무가설(두 변수는 독립이다)을 기각합니다.
       Qual

In [4]:
import itertools
import pandas as pd
from scipy import stats

def chi2_of(g):
    o = pd.crosstab(g, df["CentralAir"])
    if o.shape[0] < 2 or (o.to_numpy() == 0).any():
        return None
    return stats.chi2_contingency(o.to_numpy(), correction=False)[0]

print("--- 2구간 ---")
for k in range(1, 10):
    c = chi2_of(pd.cut(df["OverallQual"], bins=[0, k, 10]))
    if c: print([0, k, 10], round(c, 2))

print("--- 3구간 ---")
for a, b in itertools.combinations(range(1, 10), 2):
    c = chi2_of(pd.cut(df["OverallQual"], bins=[0, a, b, 10]))
    if c: print([0, a, b, 10], round(c, 2))

--- 2구간 ---
[0, 3, 10] 138.2
[0, 4, 10] 114.79
[0, 5, 10] 62.68
[0, 6, 10] 42.24
[0, 7, 10] 16.45
--- 3구간 ---
[0, 3, 4, 10] 180.76
[0, 3, 5, 10] 173.91
[0, 3, 6, 10] 166.55
[0, 3, 7, 10] 149.71
[0, 4, 5, 10] 128.38
[0, 4, 6, 10] 130.09
[0, 4, 7, 10] 121.4
[0, 5, 6, 10] 67.73
[0, 5, 7, 10] 65.03
[0, 6, 7, 10] 42.52


### 필수 1 답변 작성란

- **Q1.**
두 범주형 변수의 해당 조합에 실제로 속한 주택의 개수인 관측빈도
- **Q2.**
두 범주형 변수가 서로 독립이라는 귀무가설이참일 때 각 칸에 기대되는 빈도

- **Q3.**
65.03으로 일치
- **Q4.**
독립적이라고 보기 어려움, p-value가 0.05보다 작아 독립이라고 보기 어렵다.


---

## 필수 2. 카이제곱 독립성 검정과 비율 해석

### 문제 2-1. 건축연도 구간과 중앙 냉방시설의 관계

#### 문제 설명

건축연도를 다음 세 구간으로 나누고 중앙 냉방시설 설치 여부와 관련이 있는지 확인하세요.

- 1980년 이전
- 1980~1999년
- 2000년 이후

#### 요구사항

1. `pd.cut()`로 `YearBuiltGroup`을 만드세요.
2. `YearBuiltGroup`과 `CentralAir`의 교차표를 만드세요.
3. 다음 가설을 작성하세요.
   - H₀: 건축연도 구간과 중앙 냉방시설 유무는 서로 독립이다.
   - H₁: 건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있다.
4. 카이제곱 독립성 검정을 수행하세요.
5. 카이제곱 통계량, p-value, 자유도와 기대빈도를 출력하세요.
6. 기대빈도 중 5 미만인 칸의 개수를 확인하세요.
7. `pd.crosstab(..., normalize="index")`로 건축연도 구간별 냉방시설 비율을 계산하세요.
8. 검정 결과와 행 비율을 함께 이용해 관련성을 해석하세요.

#### 해석 질문

**Q1.** 이 문제는 적합도 검정과 독립성 검정 중 무엇을 사용해야 하나요?  
**Q2.** 자유도는 얼마이며 어떻게 계산되나요?  
**Q3.** 기대빈도 조건은 충족되나요?  
**Q4.** 건축연도 구간과 중앙 냉방시설 유무 사이에는 유의한 관련성이 있나요?  
**Q5.** 카이제곱 검정 결과만으로 건축연도가 냉방시설 설치의 원인이라고 결론 내릴 수 있나요?

#### 제출 결과

- 교차표와 가설
- 카이제곱 검정 결과
- 기대빈도 조건
- 행 비율
- 관련성 및 인과관계 해석
- Q1~Q5 답변

In [7]:
# 필수 2 코드를 작성하세요.

"""
YearBuiltGroup x CentralAir 카이제곱 독립성 검정
건축연도 구간(1980년 이전 / 1980~1999 / 2000년 이후)과
중앙 냉방시설 설치 여부가 서로 관련이 있는지 확인한다.
"""

import numpy as np
import pandas as pd
from scipy import stats


# ---------------------------------------------------------------
# 1. pd.cut() 으로 YearBuiltGroup 만들기
#    right=False -> [하한, 상한) 형태로 왼쪽 경계를 포함
#      [-inf, 1980) = 1980년 이전
#      [1980, 2000) = 1980~1999년
#      [2000,  inf) = 2000년 이후
# ---------------------------------------------------------------
bins = [-np.inf, 1980, 2000, np.inf]
labels = ["1980년 이전", "1980~1999년", "2000년 이후"]

df["YearBuiltGroup"] = pd.cut(df["YearBuilt"], bins=bins, labels=labels,
                              right=False)

print("[1] YearBuiltGroup 분포")
print(df["YearBuiltGroup"].value_counts().sort_index(), "\n")

# ---------------------------------------------------------------
# 2. 교차표 만들기
# ---------------------------------------------------------------
table = pd.crosstab(df["YearBuiltGroup"], df["CentralAir"])

print("[2] 교차표")
print(table)
print("\n    (합계 포함)")
print(pd.crosstab(df["YearBuiltGroup"], df["CentralAir"], margins=True), "\n")

# ---------------------------------------------------------------
# 3. 가설 설정
# ---------------------------------------------------------------
H0 = "H0: 건축연도 구간과 중앙 냉방시설 유무는 서로 독립이다."
H1 = "H1: 건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있다."

print("[3] 가설")
print("   ", H0)
print("   ", H1, "\n")

# ---------------------------------------------------------------
# 4. 카이제곱 독립성 검정 수행
#    correction=False : 연속성 보정 끄기 (3x2 표라 어차피 적용 안 되지만 명시)
# ---------------------------------------------------------------
observed = table.to_numpy()
chi2, p_value, dof, expected = stats.chi2_contingency(observed,
                                                      correction=False)

# ---------------------------------------------------------------
# 5. 카이제곱 통계량, p-value, 자유도, 기대빈도 출력
# ---------------------------------------------------------------
print("[5] 검정 통계량")
print(f"    카이제곱 통계량 = {chi2:.4f}")
print(f"    p-value        = {p_value:.6g}")
print(f"    자유도          = {dof}")
print("    기대빈도:")
expected_df = pd.DataFrame(expected, index=table.index, columns=table.columns)
print(expected_df.round(3), "\n")

# ---------------------------------------------------------------
# 6. 기대빈도 5 미만인 칸의 개수
# ---------------------------------------------------------------
n_below_5 = int((expected < 5).sum())
total_cells = expected.size

print("[6] 기대빈도 점검")
print(f"    5 미만인 칸: {n_below_5} / {total_cells} 개 (최소값 {expected.min():.3f})")
if n_below_5 == 0:
    print("    → 모든 칸이 5 이상이므로 카이제곱 근사를 신뢰할 수 있습니다.")
else:
    print("    → 5 미만 칸이 있어 카이제곱 근사가 부정확할 수 있습니다.")
    print("      (구간을 합치거나 Fisher 정확검정을 고려)")
print()

# ---------------------------------------------------------------
# 7. normalize="index" 로 구간별 냉방시설 비율 계산
# ---------------------------------------------------------------
row_prop = pd.crosstab(df["YearBuiltGroup"], df["CentralAir"],
                       normalize="index")

print('[7] 구간별 냉방시설 비율 (normalize="index")')
print((row_prop * 100).round(2).astype(str) + " %", "\n")

# ---------------------------------------------------------------
# 8. 검정 결과 + 행 비율로 해석
# ---------------------------------------------------------------
alpha = 0.05
overall_y = (df["CentralAir"] == "Y").mean()

print("[8] 해석")
print(f"    유의수준 alpha = {alpha}")

if p_value < alpha:
    print(f"    p-value({p_value:.3g}) < {alpha} 이므로 H0을 기각합니다.")
    print("    → 건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있습니다.\n")
else:
    print(f"    p-value({p_value:.3g}) >= {alpha} 이므로 H0을 기각하지 못합니다.")
    print("    → 두 변수가 관련 있다고 보기 어렵습니다.\n")

print(f"    전체 냉방시설 설치율: {overall_y:.1%}")
print("    구간별 설치율과 전체 대비 차이:")
for group in row_prop.index:
    rate = row_prop.loc[group, "Y"]
    gap = rate - overall_y
    print(f"      {group:<12} {rate:6.1%}  ({gap:+.1%}p)")

print("\n    p-value는 '관련이 있다'는 사실만 알려주고 방향은 알려주지 않습니다.")
print("    행 비율을 함께 보면 어느 구간이 어느 쪽으로 치우쳤는지 알 수 있습니다.")

# 기여도까지 보고 싶을 때 (어느 칸이 통계량을 만들었는지)
contrib = (observed - expected) ** 2 / expected
contrib_df = pd.DataFrame(contrib, index=table.index, columns=table.columns)
print("\n    [참고] 칸별 카이제곱 기여도")
print(contrib_df.round(3))
print(f"    합계 = {contrib.sum():.4f} (= 카이제곱 통계량)")

[1] YearBuiltGroup 분포
YearBuiltGroup
1980년 이전      848
1980~1999년    224
2000년 이후      388
Name: count, dtype: int64 

[2] 교차표
CentralAir       N    Y
YearBuiltGroup         
1980년 이전        95  753
1980~1999년       0  224
2000년 이후         0  388

    (합계 포함)
CentralAir       N     Y   All
YearBuiltGroup                
1980년 이전        95   753   848
1980~1999년       0   224   224
2000년 이후         0   388   388
All             95  1365  1460 

[3] 가설
    H0: 건축연도 구간과 중앙 냉방시설 유무는 서로 독립이다.
    H1: 건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있다. 

[5] 검정 통계량
    카이제곱 통계량 = 73.3330
    p-value        = 1.19109e-16
    자유도          = 2
    기대빈도:
CentralAir           N        Y
YearBuiltGroup                 
1980년 이전        55.178  792.822
1980~1999년      14.575  209.425
2000년 이후        25.247  362.753 

[6] 기대빈도 점검
    5 미만인 칸: 0 / 6 개 (최소값 14.575)
    → 모든 칸이 5 이상이므로 카이제곱 근사를 신뢰할 수 있습니다.

[7] 구간별 냉방시설 비율 (normalize="index")
CentralAir           N        Y
YearBuiltGroup                 
1980년 이전        1

### 필수 2 답변 작성란

- **Q1.**
독립성 검정입니다.

적합도 검정은 변수가 하나입니다. 관측 분포를 이론적 분포와 비교하죠. "CentralAir가 Y:N = 9:1일 것이다"를 검증하는 식입니다.
독립성 검정은 한 표본에서 뽑은 두 범주형 변수가 서로 관련 있는지 봅니다.
- **Q2.**
3행 2열 교차표, (3-1) *(2-1) =2
- **Q3.**

충족됩니다. 숫자를 안 봐도 확정할 수 있습니다.

기대빈도가 가장 작은 칸은 항상 N열입니다. 전체 N 비율이 95/1460 = 6.51%로 훨씬 작으니까요. 어떤 구간이든 기대 N = 구간 크기 × 0.0651이므로, 이게 5를 넘으려면 구간 크기가 77채 이상이면 됩니다.

세 구간 모두 수백 채 규모라 여유롭게 통과합니다. 6칸 전부 5 이상, 6번 요구사항의 "5 미만 칸 개수"는 0이 나옵니다.


- **Q4.**

p-value를 0.05와 비교해서 답하면 되는데, 이 데이터에선 기각 쪽이 거의 확실합니다. 중앙냉방은 1980년대 이후 신축에서 사실상 표준이 됐고 1970년대 이전 주택엔 드물어서, 구간별 설치율 격차가 클 수밖에 없습니다.

다만 답안에는 실행 결과의 실제 숫자를 넣으세요. 이런 형태가 좋습니다.

카이제곱 통계량 ○○○, 자유도 2, p-value ○○○로 유의수준 0.05에서 H₀를 기각한다. 즉 건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있다. 행 비율을 보면 1980년 이전 구간의 설치율이 ○○%인 반면 2000년 이후는 ○○%로, 최근에 지어진 주택일수록 설치율이 높다.

통계량만 쓰고 끝내면 감점 요인입니다. 8번 요구사항이 "검정 결과와 행 비율을 함께 이용해 해석하라"인 이유가 이겁니다. p-value는 관련성의 존재만 알려주고 방향은 못 알려줍니다.

효과 크기를 덧붙이려면 Cramér's V = √(χ²/n)을 쓰면 됩니다. 2×k 표에선 min(r−1, c−1) = 1이라 그냥 √(χ²/1460)입니다.

- **Q5.**

카이제곱 독립성 검정은 두 범주형 변수 간 통계적 연관성의 유무만 검정할 뿐 인과관계를 확인하는 방법이 아니며, 관찰 데이터에서는 시기별 건축 기준 변화나 리모델링 이력 같은 교란 요인이 통제되지 않으므로 인과 결론을 내릴 수 없다.


---

## 과제. 주방 품질과 진입로 포장 상태의 관계

### 문제 3-1. 두 범주형 변수 재분류 후 독립성 검정

#### 문제 설명

기대빈도가 너무 작은 범주를 줄이기 위해 주방 품질과 진입로 포장 상태를 다음과 같이 재분류합니다.

- `KitchenGroup`
  - 우수: `Ex`, `Gd`
  - 보통 이하: `TA`, `Fa`
- `DriveGroup`
  - 완전 포장: `Y`
  - 미포장·부분포장: `N`, `P`

#### 요구사항

1. 위 기준으로 `KitchenGroup`과 `DriveGroup`을 만드세요.
2. 두 변수의 교차표를 작성하세요.
3. 두 변수가 독립이라는 귀무가설과 관련이 있다는 대립가설을 작성하세요.
4. 카이제곱 독립성 검정을 수행하세요.
5. 카이제곱 통계량, p-value, 자유도와 기대빈도를 출력하세요.
6. 모든 기대빈도가 5 이상인지 확인하세요.
7. 주방 품질 집단별 진입로 포장 비율을 계산하세요.
8. 검정 결과와 비율 차이를 함께 사용하여 두 변수의 관련성을 해석하세요.

#### 해석 질문

**Q1.** 이 과제에서 범주를 재분류한 이유는 무엇인가요?  
**Q2.** 기대빈도 조건은 충족되나요?  
**Q3.** 주방 품질 집단과 진입로 포장 상태는 서로 독립이라고 볼 수 있나요?  
**Q4.** 두 주방 품질 집단의 완전 포장 비율은 각각 얼마인가요?

#### 제출 결과

- 재분류 코드와 교차표
- 가설 설정
- 카이제곱 검정 결과
- 기대빈도 조건 확인
- 행 비율과 결과 해석
- Q1~Q4 답변

In [2]:
# 과제 코드를 작성하세요.

import pandas as pd
from scipy import stats


# 1. KitchenGroup, DriveGroup 생성
kitchen_map = {"Ex": "우수", "Gd": "우수", "TA": "보통이하", "Fa": "보통이하"}
drive_map = {"Y": "완전포장", "N": "미포장·부분포장", "P": "미포장·부분포장"}

df["KitchenGroup"] = df["KitchenQual"].map(kitchen_map)
df["DriveGroup"] = df["PavedDrive"].map(drive_map)

# 매핑에서 빠진 값(결측 등) 제거
data = df.dropna(subset=["KitchenGroup", "DriveGroup"])

# 2. 교차표
ct = pd.crosstab(data["KitchenGroup"], data["DriveGroup"])
print("교차표")
print(ct)

# 3. 가설
# H0: 주방 품질과 진입로 포장 상태는 서로 독립이다. (관련이 없다)
# H1: 주방 품질과 진입로 포장 상태는 독립이 아니다. (관련이 있다)

alpha = 0.05

# 4~5. 카이제곱 독립성 검정
chi2, p_value, dof, expected = stats.chi2_contingency(ct)

print(f"\n카이제곱 통계량: {chi2:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"자유도: {dof}")
print("\n기대빈도")
print(pd.DataFrame(expected, index=ct.index, columns=ct.columns).round(4))

# 6. 모든 기대빈도가 5 이상인지 확인
min_expected = expected.min()
print(f"\n최소 기대빈도: {min_expected:.4f}")
if min_expected >= 5:
    print("모든 기대빈도가 5 이상 → 카이제곱 검정 사용 조건을 충족합니다.")
else:
    print("기대빈도가 5 미만인 칸이 있음 → Fisher 정확검정 등을 고려해야 합니다.")

# 7. 주방 품질 집단별 진입로 포장 비율 (행 기준 비율)
ratio = pd.crosstab(data["KitchenGroup"], data["DriveGroup"], normalize="index")
print("\n주방 품질 집단별 진입로 포장 비율")
print((ratio * 100).round(2))

# 8. 검정 결과 + 비율 차이 해석
if p_value < alpha:
    print(f"\np-value < {alpha} → 귀무가설 기각: 주방 품질과 진입로 포장 상태는 서로 관련이 있습니다.")
else:
    print(f"\np-value >= {alpha} → 귀무가설 기각 못함: 두 변수가 관련 있다고 볼 근거가 부족합니다.")

paved_ratio = ratio["완전포장"]
print(f"완전포장 비율 - 우수: {paved_ratio['우수']*100:.2f}%, "
      f"보통이하: {paved_ratio['보통이하']*100:.2f}%")
print(f"비율 차이: {abs(paved_ratio['우수'] - paved_ratio['보통이하'])*100:.2f}%p")

교차표
DriveGroup    미포장·부분포장  완전포장
KitchenGroup                
보통이하               100   674
우수                  20   666

카이제곱 통계량: 46.9352
p-value: 0.0000
자유도: 1

기대빈도
DriveGroup    미포장·부분포장      완전포장
KitchenGroup                    
보통이하           63.6164  710.3836
우수             56.3836  629.6164

최소 기대빈도: 56.3836
모든 기대빈도가 5 이상 → 카이제곱 검정 사용 조건을 충족합니다.

주방 품질 집단별 진입로 포장 비율
DriveGroup    미포장·부분포장   완전포장
KitchenGroup                 
보통이하             12.92  87.08
우수                2.92  97.08

p-value < 0.05 → 귀무가설 기각: 주방 품질과 진입로 포장 상태는 서로 관련이 있습니다.
완전포장 비율 - 우수: 97.08%, 보통이하: 87.08%
비율 차이: 10.00%p


### 과제 답변 작성란


**Q1.** 이 과제에서 범주를 재분류한 이유는 무엇인가요?  
기대 빈도가 너무 작은 범주가 생기면 카이제곱 저멎ㅇ의 근사가 정확하지 않기 때문에, 빈도가 적은 범주 를 비슷한 성격끼리 묶어 2*2 교차표로 만드렁ㅆ고 그 결과 최소 기대빈도가 56.38이 되어 검정 조건 을 충족

**Q2.** 기대빈도 조건은 충족되나요?  
최소 기대 빈대는 56.38 그리고 모든 기대빈도가 5이상이므로 카이제곱 검증 통과

**Q3.** 주방 품질 집단과 진입로 포장 상태는 서로 독립이라고 볼 수 있나요?  
p value 가 0.05보다 작아서 귀무가설  기각 이고 서로 독립 변수로 못본다.

**Q4.** 두 주방 품질 집단의 완전 포장 비율은 각각 얼마인가요?
완전 포장 비율은 보통이하는 87.88 우수는 97.08이다.

---

## 실습 마무리

1. 교차표에서 관측빈도와 기대빈도는 어떻게 다른가요?

관측빈도는 데이터에서 실제로 센 개수, 기대빈도는 두 변수가 독립이라고 가정할때 주 변합으로 예상되는개수

2. 기대빈도는 어떤 공식으로 계산하나요?
-> (해당 행 * 해당 열) % 전체 합계

3. 카이제곱 통계량이 커진다는 것은 무엇을 의미하나요?
-> 관측빈도가 독립을 가정한 기대빈도에서 전체적으로 더 크게 벗어난다.

4. 독립성 검정과 적합도 검정은 변수 개수와 질문에서 어떻게 다른가요?
독립성 검정은 두 범주형 변수의 관계를 확인하고, 적합도 검정은 하나의 범주형 변수의 관측 분포가 기대한 분포와 일치하는지 확인한다.

5. 기대빈도가 5보다 작은 칸이 있다면 무엇을 고려해야 하나요?
-> 기대 빈도가 작다면 카이제곱 분포로 근사한 p-value의 정확성이 떨어 질 수 있다.
